# RESUME — `v28-resnet_style_128`

Picks up the run that was killed at **epoch 13**, from its checkpoint on Drive. It does not
start over.

Where it was when it died:

| | epoch | macro-F1 | Scratch |
|---|---|---|---|
| `v28-resnet_style_128` | 13 | 0.8878 | 0.800 |

`Scratch` 0.800 is the second-highest recorded in this project, behind `v30`'s 0.813, and it
was still rising — 0.686 at epoch 9, 0.800 at 13. The earlier read that it "wasn't going
well" came from the epoch-9 number.

This is `v27-resnet_style` (0.8900, `Scratch` 0.759 at 64px) run at 128px, everything else
identical, so the pair differs in input resolution alone.

## Before you start

1. `git pull` in `/content/fdl-project`, then **Runtime > Restart session**.
2. `wandb login`, or `WANDB_KEY` in Colab Secrets.
3. Run All. Section 3 refuses to proceed if no checkpoint is found, so a silent restart
   from epoch 1 cannot happen by accident.

## 0. Colab web UI only — clone and authenticate

Skip if `/content/fdl-project` already exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Confirm there is something to resume from

The failure this guards against is silent: with no checkpoint the runner starts at epoch 1
and looks completely normal for the two hours it takes to get back to where it already was.
This cell lists what is on Drive and reads the epoch out of the newest file.

In [ ]:
from pathlib import Path

import torch

SERIES = "v28_capacity"
RUN_NAME = "v28-resnet_style_128"

candidates = [CHECKPOINTS / RUN_NAME] if HAS_DRIVE else []
candidates.append(REPO / "trained-models/checkpoints" / RUN_NAME)

found = None
for directory in candidates:
    if directory.is_dir():
        files = sorted(directory.glob("*.pt"))
        if files:
            found = directory
            print(f"checkpoints in {directory}:")
            for path in files:
                size = path.stat().st_size / 1024**2
                print(f"  {path.name:28} {size:7.1f} MiB")
            break
    print(f"  nothing in {directory}")

assert found is not None, (
    "No checkpoint found. Without one this notebook would silently restart from "
    "epoch 1 rather than resume, so it stops here instead."
)

# Read the epoch out of the newest checkpoint, so you know what you are resuming
# from before an hour goes into it.
newest = max(found.glob("*.pt"), key=lambda p: p.stat().st_mtime)
state = torch.load(newest, map_location="cpu", weights_only=False)
epoch = state.get("epoch") if isinstance(state, dict) else None
print(f"\nnewest: {newest.name}")
print(f"resuming from epoch {epoch}" if epoch else
      "could not read an epoch from the checkpoint; the runner will still resume")

## 4. Resume

`checkpoint.resume=auto` is passed explicitly and asserted. The run keeps its original W&B
name, so its history continues rather than forking.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

CONFIG = REPO / "configs/train" / SERIES / "02_resnet_style_128.yaml"
assert CONFIG.exists(), f"{CONFIG} missing -- git pull, then Runtime > Restart session"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = ["data.transform_device=cuda", "checkpoint.resume=auto"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},resumed]"]

config = load_experiment_config(CONFIG, overrides=OVERRIDES)
assert config.name == RUN_NAME, f"expected {RUN_NAME}, got {config.name}"
assert config.checkpoint.resume == "auto", "resume must be on or this restarts from scratch"

model = build_model(config.model.name, **config.model.kwargs)
print(f"=== {config.name}  ({count_trainable_parameters(model):,} parameters, "
      f"{config.data.preprocessing.target_size[0]}px)")
print(f"    max_epochs {config.trainer.max_epochs}, patience "
      f"{config.trainer.early_stopping.patience}, resume {config.checkpoint.resume}")
del model

dataframe = load_wm811k_dataframe(DATASET)
started = time.monotonic()
result = run_experiment(config, overwrite=True, dataframe=dataframe)

macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
row = {
    "run": config.name,
    "px": config.data.preprocessing.target_size[0],
    "macro_f1": round(float(macro.point_estimate), 4),
    "ci_lower": round(float(macro.ci_lower), 4),
    "ci_upper": round(float(macro.ci_upper), 4),
    "scratch_f1": round(float(per_class["Scratch"]), 3),
    "near_full_f1": round(float(per_class["Near-full"]), 3),
    "best_epoch": result.fit.best_epoch,
    "epochs": len(result.fit.history),
    "minutes": round((time.monotonic() - started) / 60, 1),
}
csv = OUTPUT / "resnet_style_128_results.csv"
pd.DataFrame([row]).to_csv(csv, index=False)
if HAS_DRIVE:
    shutil.copy2(csv, DRIVE / f"{SERIES}_resnet_style_128.csv")

print(f"\n  macro-F1 {row['macro_f1']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
print(f"  Scratch {row['scratch_f1']:.3f}   Near-full {row['near_full_f1']:.3f}")
print(f"  best epoch {row['best_epoch']}/{row['epochs']}   {row['minutes']:.1f} min")
if row["best_epoch"] >= row["epochs"] - 2:
    print("  STILL IMPROVING AT THE CAP -- this number is a floor.")
print(f"  saved to {csv}")

## 5. Where it lands

Run after section 4 finishes.

In [ ]:
REFERENCE = [
    ("v32-resnet34_finetune",      "pretrained 21.4M, 128px", 0.9041, 0.793),
    ("dilated-style-64-dihedral8", "ours 298k, old aug",      0.8995, 0.793),
    ("v28-convnext_big_128",       "ours 2.68M, 128px",       0.8988, 0.787),
    ("v30-finetune_encoder_1e-5",  "pretrained 11.3M, 128px", 0.8938, 0.813),
    ("v27-resnet_style",           "ours 2.83M, 64px",        0.8900, 0.759),
    ("v28-convnext_big_64",        "ours 2.68M, 64px",        0.8862, 0.740),
    ("v27-baseline_cnn",           "ours 157k, 64px",         0.8646, 0.715),
]
NOISE_FLOOR = 0.02

table = pd.DataFrame(REFERENCE, columns=["run", "what", "macro_f1", "scratch_f1"])
mine = pd.DataFrame([{"run": row["run"], "what": "THIS NOTEBOOK (ours 2.83M, 128px)",
                      "macro_f1": row["macro_f1"], "scratch_f1": row["scratch_f1"]}])
pd.set_option("display.width", 210)
display(pd.concat([mine, table]).sort_values("macro_f1", ascending=False).reset_index(drop=True))

# The pairing this arm is for: the same model at 64px.
delta = row["macro_f1"] - 0.8900
verdict = "REAL" if abs(delta) > NOISE_FLOOR else "inside the noise floor"
print(f"vs v27-resnet_style, the same model at 64px (0.8900): {delta:+.4f}  [{verdict}]")
print(f"Scratch: {row['scratch_f1']:.3f} against 0.759 at 64px")

## 6. Report back

```
resnet_style_128 (resumed): macro-F1 X.XXXX [lo, hi], Scratch X.XXX, best epoch N/M
```

CSV at `output/v28_capacity/resnet_style_128_results.csv`, copied to Drive.